# 04 — GLO Baseline Machine Learning

**Project:** GlacierGuard-AI  
**Task:** Next-Year Glacial-Lake Area Change Benchmark Prediction  
**Primary Research Target:** `NEXT_AREA_CHANGE = AREA(t+1) - AREA(t)` (in $\text{km}^2$, where $\Delta t = 1$)  
**Source Data:** Authoritative Gold Datasets from Notebook 03 (`gold_s1`, `gold_s2`)  

---

### Scientific Scope & Boundaries
1. **Prediction Task:** This experiment models next-year annual lake area change ($\text{km}^2$). It is **NOT** a Glacial Lake Outburst Flood (GLOF) prediction model, and must **NOT** be described as hazard prediction, flood forecasting, or operational early warning.
2. **Empirical Objective:** Rigorously establish whether information available at year $t$ (geomorphology, location, backward area lags, backward changes, backward rolling stats) carries measurable predictive signal over simple baseline benchmarks.
3. **Chronological Splitting:** Training ($\le 2021$), Validation ($2022$), and Testing ($2023$) respect temporal ordering without random data leakage.
4. **Sensor Independence:** Sentinel-1 (SAR) and Sentinel-2 (Optical) feature datasets are modeled independently to reflect their distinct physical acquisition characteristics.
5. **Zero Data Fabrication:** Missing historical observation years remain `NaN` and are imputed strictly through training-fitted scikit-learn pipelines.

## 2. Imports & Configuration

In [1]:
# == 2.1  Imports, Environment, and Reproducibility Configuration ==
import os
import sys
import pathlib
import warnings

# Headless matplotlib backend
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import geopandas as gpd

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Reproducibility and CPU settings
RANDOM_STATE = 42
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
warnings.filterwarnings('ignore', category=UserWarning)

# Report and figure directory
FIGURES_DIR = pathlib.Path('../reports/figures')
if not FIGURES_DIR.exists():
    FIGURES_DIR = pathlib.Path('reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Environment Configuration:')
print(f'  Python       : {sys.version.split()[0]}')
print(f'  scikit-learn : {sklearn.__version__}')
print(f'  pandas       : {pd.__version__}')
print(f'  numpy        : {np.__version__}')
print(f'  matplotlib   : {matplotlib.__version__}')
print(f'  Figures path : {FIGURES_DIR.resolve()}')
print(f'  Random Seed  : {RANDOM_STATE}')


Environment Configuration:
  Python       : 3.12.14
  scikit-learn : 1.6.1
  pandas       : 2.2.3
  numpy        : 2.1.3
  matplotlib   : 3.10.9
  Figures path : C:\Users\ASUS\Desktop\GlacierGuard-AI\reports\figures
  Random Seed  : 42


## 3. Load Gold Datasets

Notebook 03 is the **authoritative feature-engineering source**. Here we replicate the validated Notebook 03 pipeline in pure Pandas directly from the raw annual GeoPackages to instantiate `gold_s1` and `gold_s2`.

In [2]:
# == 3.1  Instantiate Authoritative Gold Datasets ==
GLO_DIR_candidates = [
    pathlib.Path('../raw/GLO'),
    pathlib.Path('./raw/GLO'),
    pathlib.Path('/Workspace/raw/GLO'),
]

GLO_DIR = None
for p in GLO_DIR_candidates:
    if p.exists():
        GLO_DIR = p.resolve()
        break

if GLO_DIR is None:
    raise FileNotFoundError('Cannot locate raw/GLO directory.')

def build_gold_dataset(gpkg_path):
    df = gpd.read_file(str(gpkg_path)).drop(columns='geometry')
    df['AREA_YEAR'] = df['AREA_YEAR'].astype(int)
    df = df.sort_values(['GLO_ID', 'AREA_YEAR']).reset_index(drop=True)
    
    g = df.groupby('GLO_ID')
    df['PREV_YEAR'] = g['AREA_YEAR'].shift(1)
    df['PREV_AREA'] = g['AREA'].shift(1)
    df['PREV_YEAR_GAP'] = df['AREA_YEAR'] - df['PREV_YEAR']
    df['PREV_YEAR_2'] = g['AREA_YEAR'].shift(2)
    df['PREV_AREA_2'] = g['AREA'].shift(2)
    df['PREV_YEAR_3'] = g['AREA_YEAR'].shift(3)
    df['PREV_AREA_3'] = g['AREA'].shift(3)
    df['PREV_YEAR_4'] = g['AREA_YEAR'].shift(4)
    df['PREV_AREA_4'] = g['AREA'].shift(4)
    df['NEXT_YEAR'] = g['AREA_YEAR'].shift(-1)
    df['NEXT_AREA'] = g['AREA'].shift(-1)
    df['NEXT_YEAR_GAP'] = df['NEXT_YEAR'] - df['AREA_YEAR']
    
    # Gap-enforced lags
    df['AREA_LAG1'] = np.where(df['PREV_YEAR_GAP'] == 1, df['PREV_AREA'], np.nan)
    df['AREA_LAG2'] = np.where((df['PREV_YEAR_GAP'] == 1) & ((df['PREV_YEAR'] - df['PREV_YEAR_2']) == 1), df['PREV_AREA_2'], np.nan)
    df['AREA_LAG3'] = np.where((df['PREV_YEAR_GAP'] == 1) & ((df['PREV_YEAR'] - df['PREV_YEAR_2']) == 1) & ((df['PREV_YEAR_2'] - df['PREV_YEAR_3']) == 1), df['PREV_AREA_3'], np.nan)
    df['AREA_LAG4'] = np.where((df['PREV_YEAR_GAP'] == 1) & ((df['PREV_YEAR'] - df['PREV_YEAR_2']) == 1) & ((df['PREV_YEAR_2'] - df['PREV_YEAR_3']) == 1) & ((df['PREV_YEAR_3'] - df['PREV_YEAR_4']) == 1), df['PREV_AREA_4'], np.nan)
    
    # Consecutive historical changes
    df['AREA_CHANGE'] = np.where(df['PREV_YEAR_GAP'] == 1, df['AREA'] - df['PREV_AREA'], np.nan)
    df['AREA_CHANGE_LAG1'] = np.where(df['AREA_LAG1'].notna() & df['AREA_LAG2'].notna(), df['AREA_LAG1'] - df['AREA_LAG2'], np.nan)
    df['AREA_CHANGE_LAG2'] = np.where(df['AREA_LAG2'].notna() & df['AREA_LAG3'].notna(), df['AREA_LAG2'] - df['AREA_LAG3'], np.nan)
    df['AREA_CHANGE_LAG3'] = np.where(df['AREA_LAG3'].notna() & df['AREA_LAG4'].notna(), df['AREA_LAG3'] - df['AREA_LAG4'], np.nan)
    
    # Backward-looking rolling statistics
    c_df = df[['AREA_CHANGE_LAG1', 'AREA_CHANGE_LAG2', 'AREA_CHANGE_LAG3']]
    df['ROLLING_MEAN_3'] = c_df.mean(axis=1, skipna=True)
    df['ROLLING_STD_3'] = c_df.std(axis=1, skipna=True, ddof=1)
    
    # Supervised prediction target
    df['NEXT_AREA_CHANGE'] = np.where(df['NEXT_YEAR_GAP'] == 1, df['NEXT_AREA'] - df['AREA'], np.nan)
    df['IS_TS_OUTLIER'] = df['TS_OUTLIER'].astype(str).str.upper().isin(['TRUE', '1'])
    
    # Select official Gold columns (dropping temporary/intermediate fields like AREA_LAG4, NEXT_AREA, etc.)
    gold_cols = [
        'GLO_ID', 'AREA_YEAR', 'AREA', 'PERIMETER', 'AREA_UNCERTAINTY',
        'ELEVATION_MEAN', 'ELEVATION_MIN', 'ELEVATION_MEDIAN',
        'CONNECTIVITY', 'LATITUDE', 'LONGITUDE', 'BASIN', 'COUNTRY', 'GTNG_REGION_O2',
        'TS_OUTLIER', 'IS_TS_OUTLIER',
        'AREA_LAG1', 'AREA_LAG2', 'AREA_LAG3',
        'AREA_CHANGE_LAG1', 'AREA_CHANGE_LAG2', 'AREA_CHANGE_LAG3',
        'ROLLING_MEAN_3', 'ROLLING_STD_3',
        'AREA_CHANGE',  # Preserved strictly for persistence benchmark
        'NEXT_AREA_CHANGE'
    ]
    return df[gold_cols].copy()

gold_s1 = build_gold_dataset(GLO_DIR / 'S1_20172024_NTB_GLOID_v1.02.gpkg')
gold_s2 = build_gold_dataset(GLO_DIR / 'S2_20172024_NTB_GLOID_v1.02.gpkg')

# Verify contract counts against Notebook 03
assert len(gold_s1) == 18129, f'Expected 18,129 S1 rows, got {len(gold_s1)}'
assert len(gold_s2) == 22294, f'Expected 22,294 S2 rows, got {len(gold_s2)}'
assert gold_s1['NEXT_AREA_CHANGE'].notna().sum() == 14242, 'S1 supervised target mismatch'
assert gold_s2['NEXT_AREA_CHANGE'].notna().sum() == 16610, 'S2 supervised target mismatch'

print(f'✓ gold_s1 instantiated: {len(gold_s1):,} rows, {gold_s1["NEXT_AREA_CHANGE"].notna().sum():,} supervised targets')
print(f'✓ gold_s2 instantiated: {len(gold_s2):,} rows, {gold_s2["NEXT_AREA_CHANGE"].notna().sum():,} supervised targets')


✓ gold_s1 instantiated: 18,129 rows, 14,242 supervised targets
✓ gold_s2 instantiated: 22,294 rows, 16,610 supervised targets


## 4. Validate Gold Schema

In [3]:
# == 4.1  Schema & Leakage Verification ==
PROHIBITED_LEAKAGE = [
    'NEXT_AREA', 'NEXT_YEAR', 'NEXT_YEAR_GAP',
    'EXPANSION_RATE', 'EXPANSION_RATE_SIG', 'EXPANSION_UNCERTAINTY',
    'START_DATE', 'END_DATE', 'REF_MSTAT', 'S2_MSTAT', 'DATA_SOURCE',
    'AREA_LAG4'
]

for label, df in [('Sentinel-1', gold_s1), ('Sentinel-2', gold_s2)]:
    cols = set(df.columns)
    leaked = [c for c in PROHIBITED_LEAKAGE if c in cols]
    assert len(leaked) == 0, f'Leakage detected in {label}: {leaked}'

print('✓ Schema verified: zero prohibited or intermediate leakage columns present.')


✓ Schema verified: zero prohibited or intermediate leakage columns present.


## 5. Define Model Feature Contract

Exactly 20 candidate features are approved for predictive modeling. Categorical variables are separated from numerical variables to structure the preprocessing pipeline.

In [4]:
# == 5.1  Candidate Feature Specification ==
CATEGORICAL_FEATURES = ['BASIN', 'COUNTRY', 'GTNG_REGION_O2', 'CONNECTIVITY']
NUMERICAL_FEATURES = [
    'AREA', 'PERIMETER', 'AREA_UNCERTAINTY',
    'ELEVATION_MEAN', 'ELEVATION_MIN', 'ELEVATION_MEDIAN',
    'LATITUDE', 'LONGITUDE',
    'AREA_LAG1', 'AREA_LAG2', 'AREA_LAG3',
    'AREA_CHANGE_LAG1', 'AREA_CHANGE_LAG2', 'AREA_CHANGE_LAG3',
    'ROLLING_MEAN_3', 'ROLLING_STD_3'
]

MODEL_FEATURE_VECTOR = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
TARGET_COL = 'NEXT_AREA_CHANGE'

assert len(MODEL_FEATURE_VECTOR) == 20, f'Expected 20 features, got {len(MODEL_FEATURE_VECTOR)}'
assert TARGET_COL not in MODEL_FEATURE_VECTOR, 'Target cannot be in feature vector'
assert 'GLO_ID' not in MODEL_FEATURE_VECTOR, 'GLO_ID must not be a predictive feature'
assert 'AREA_YEAR' not in MODEL_FEATURE_VECTOR, 'AREA_YEAR must not be a predictive feature'
assert 'TS_OUTLIER' not in MODEL_FEATURE_VECTOR, 'TS_OUTLIER excluded from primary features'
assert 'IS_TS_OUTLIER' not in MODEL_FEATURE_VECTOR, 'IS_TS_OUTLIER excluded from primary features'
assert 'AREA_CHANGE' not in MODEL_FEATURE_VECTOR, 'Current AREA_CHANGE excluded from ML feature vector'

print(f'Model Feature Contract finalized: {len(MODEL_FEATURE_VECTOR)} features')
print(f'  Categorical ({len(CATEGORICAL_FEATURES)}) : {CATEGORICAL_FEATURES}')
print(f'  Numerical   ({len(NUMERICAL_FEATURES)}) : {NUMERICAL_FEATURES}')
print(f'  Target                     : {TARGET_COL}')


Model Feature Contract finalized: 20 features
  Categorical (4) : ['BASIN', 'COUNTRY', 'GTNG_REGION_O2', 'CONNECTIVITY']
  Numerical   (16) : ['AREA', 'PERIMETER', 'AREA_UNCERTAINTY', 'ELEVATION_MEAN', 'ELEVATION_MIN', 'ELEVATION_MEDIAN', 'LATITUDE', 'LONGITUDE', 'AREA_LAG1', 'AREA_LAG2', 'AREA_LAG3', 'AREA_CHANGE_LAG1', 'AREA_CHANGE_LAG2', 'AREA_CHANGE_LAG3', 'ROLLING_MEAN_3', 'ROLLING_STD_3']
  Target                     : NEXT_AREA_CHANGE


## 6. Temporal Train / Validation / Test Split

No random train/test splitting. Partitions follow strict chronological ordering:
- **TRAIN:** Current observation years 2017–2021 (target corresponds to 2018–2022).
- **VALIDATION:** Current observation year 2022 (target corresponds to 2023).
- **TEST:** Current observation year 2023 (target corresponds to 2024).
- Terminal observation year 2024 is excluded from supervised learning as no 2025 target exists.

In [5]:
# == 6.1  Chronological Temporal Partitions ==
def create_temporal_splits(df, label):
    sup = df[df[TARGET_COL].notna()].copy()
    train = sup[sup['AREA_YEAR'] <= 2021].copy()
    val = sup[sup['AREA_YEAR'] == 2022].copy()
    test = sup[sup['AREA_YEAR'] == 2023].copy()
    
    assert len(train) + len(val) + len(test) == len(sup), 'Partition row mismatch'
    assert train['AREA_YEAR'].max() <= 2021, 'Train year violation'
    assert (val['AREA_YEAR'] == 2022).all(), 'Val year violation'
    assert (test['AREA_YEAR'] == 2023).all(), 'Test year violation'
    assert (sup['AREA_YEAR'] >= 2024).sum() == 0, 'Post-2023 found in supervised data'
    
    print(f'\n{label} Temporal Splits:')
    print(f'  Train      (<=2021): {len(train):,} rows [min={train["AREA_YEAR"].min()}, max={train["AREA_YEAR"].max()}]')
    print(f'  Validation (==2022): {len(val):,} rows [min={val["AREA_YEAR"].min()}, max={val["AREA_YEAR"].max()}]')
    print(f'  Test       (==2023): {len(test):,} rows [min={test["AREA_YEAR"].min()}, max={test["AREA_YEAR"].max()}]')
    print(f'  Total Supervised   : {len(sup):,} rows')
    return train, val, test

s1_train, s1_val, s1_test = create_temporal_splits(gold_s1, 'Sentinel-1')
s2_train, s2_val, s2_test = create_temporal_splits(gold_s2, 'Sentinel-2')

# Numerical contract assertions
assert len(s1_train) == 10161 and len(s1_val) == 2026 and len(s1_test) == 2055
assert len(s2_train) == 11858 and len(s2_val) == 2245 and len(s2_test) == 2507
print('✓ Chronological split counts verified with 100% parity to Notebook 03.')



Sentinel-1 Temporal Splits:
  Train      (<=2021): 10,161 rows [min=2017, max=2021]
  Validation (==2022): 2,026 rows [min=2022, max=2022]
  Test       (==2023): 2,055 rows [min=2023, max=2023]
  Total Supervised   : 14,242 rows

Sentinel-2 Temporal Splits:
  Train      (<=2021): 11,858 rows [min=2017, max=2021]
  Validation (==2022): 2,245 rows [min=2022, max=2022]
  Test       (==2023): 2,507 rows [min=2023, max=2023]
  Total Supervised   : 16,610 rows
✓ Chronological split counts verified with 100% parity to Notebook 03.


## 7. Target & Feature Preparation

In [6]:
# == 7.1  Assemble Matrices X and Targets y ==
X_s1_train, y_s1_train = s1_train[MODEL_FEATURE_VECTOR], s1_train[TARGET_COL]
X_s1_val, y_s1_val = s1_val[MODEL_FEATURE_VECTOR], s1_val[TARGET_COL]
X_s1_test, y_s1_test = s1_test[MODEL_FEATURE_VECTOR], s1_test[TARGET_COL]

X_s2_train, y_s2_train = s2_train[MODEL_FEATURE_VECTOR], s2_train[TARGET_COL]
X_s2_val, y_s2_val = s2_val[MODEL_FEATURE_VECTOR], s2_val[TARGET_COL]
X_s2_test, y_s2_test = s2_test[MODEL_FEATURE_VECTOR], s2_test[TARGET_COL]

# Sanity assertions
for split_X, split_y in [(X_s1_train, y_s1_train), (X_s1_val, y_s1_val), (X_s1_test, y_s1_test),
                        (X_s2_train, y_s2_train), (X_s2_val, y_s2_val), (X_s2_test, y_s2_test)]:
    assert split_X.shape[1] == 20, f'Expected 20 features, got {split_X.shape[1]}'
    assert not split_y.isna().any(), 'Target contains unexpected NaN'
    assert TARGET_COL not in split_X.columns, 'Target leaked into X'

print('✓ Feature matrices X and targets y prepared for all partitions.')


✓ Feature matrices X and targets y prepared for all partitions.


## 8. Preprocessing Pipeline

Preprocessing is fitted **strictly on training data** via scikit-learn `Pipeline` and `ColumnTransformer`:
- **Numerical:** `SimpleImputer(strategy='median')`
- **Categorical:** `SimpleImputer(strategy='most_frequent')` $\to$ `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`

In [7]:
# == 8.1  Construct Leak-Free Preprocessor ==
def build_preprocessor():
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', num_pipe, NUMERICAL_FEATURES),
        ('cat', cat_pipe, CATEGORICAL_FEATURES)
    ])

print('✓ Preprocessing pipeline definition constructed.')


✓ Preprocessing pipeline definition constructed.


## 9. Baseline Benchmarks

Two simple benchmarks are evaluated before training ML algorithms:
1. **Mean Baseline:** Predicts training set target mean ($\bar{y}_{\text{train}}$) for all observations.
2. **Persistence Change Baseline:** Predicts most recent observed annual change ($\text{AREA\_CHANGE} = A_t - A_{t-1}$). If $\text{AREA\_CHANGE}$ is missing/NaN, falls back to $\bar{y}_{\text{train}}$.

In [8]:
# == 9.1  Evaluate Baseline Benchmarks ==
results_records = []

def evaluate_predictions(y_true, y_pred, sensor, model_name, split_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rec = {
        'Sensor': sensor,
        'Model': model_name,
        'Split': split_name,
        'N': len(y_true),
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Target_Mean': y_true.mean(),
        'Target_Median': y_true.median()
    }
    results_records.append(rec)
    return rec

for sensor_lbl, train_df, val_df, test_df in [('Sentinel-1', s1_train, s1_val, s1_test), ('Sentinel-2', s2_train, s2_val, s2_test)]:
    y_tr = train_df[TARGET_COL]
    y_va = val_df[TARGET_COL]
    y_te = test_df[TARGET_COL]
    tr_mean = y_tr.mean()
    
    # Mean Baseline
    for split_lbl, yt in [('Train', y_tr), ('Validation', y_va), ('Test', y_te)]:
        pred_mean = np.full(len(yt), tr_mean)
        evaluate_predictions(yt, pred_mean, sensor_lbl, 'Mean Baseline', split_lbl)
        
    # Persistence Change Baseline (AREA_CHANGE = AREA - AREA_LAG1)
    for split_lbl, d, yt in [('Train', train_df, y_tr), ('Validation', val_df, y_va), ('Test', test_df, y_te)]:
        pred_persist = np.where(d['AREA_CHANGE'].notna(), d['AREA_CHANGE'], tr_mean)
        evaluate_predictions(yt, pred_persist, sensor_lbl, 'Change Persistence Baseline', split_lbl)

print('✓ Baseline benchmarks evaluated.')


✓ Baseline benchmarks evaluated.


## 10–13. Supervised Machine Learning Models

We train four supervised models on each sensor's training split:
1. **Linear Regression** (unregularized OLS)
2. **Ridge Regression** ($L_2$ regularization, $\alpha=1.0$)
3. **Random Forest Regressor** (100 trees, max depth 10, min leaf 5)
4. **HistGradientBoosting Regressor** (100 iterations, max depth 6, min leaf 20)

In [9]:
# == 10.1  Train and Benchmark Supervised Models ==
trained_models = {}
predictions_store = {}

model_constructors = {
    'Linear Regression': lambda: LinearRegression(),
    'Ridge': lambda: Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest': lambda: RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1),
    'HistGradientBoosting': lambda: HistGradientBoostingRegressor(max_iter=100, max_depth=6, min_samples_leaf=20, random_state=RANDOM_STATE)
}

for sensor_lbl, (X_tr, y_tr), (X_va, y_va), (X_te, y_te) in [
    ('Sentinel-1', (X_s1_train, y_s1_train), (X_s1_val, y_s1_val), (X_s1_test, y_s1_test)),
    ('Sentinel-2', (X_s2_train, y_s2_train), (X_s2_val, y_s2_val), (X_s2_test, y_s2_test))
]:
    for m_name, m_ctor in model_constructors.items():
        pipe = Pipeline([
            ('preprocessor', build_preprocessor()),
            ('regressor', m_ctor())
        ])
        
        pipe.fit(X_tr, y_tr)
        key = f'{sensor_lbl}_{m_name}'
        trained_models[key] = pipe
        
        p_tr = pipe.predict(X_tr)
        p_va = pipe.predict(X_va)
        p_te = pipe.predict(X_te)
        
        predictions_store[f'{key}_val'] = p_va
        predictions_store[f'{key}_test'] = p_te
        
        evaluate_predictions(y_tr, p_tr, sensor_lbl, m_name, 'Train')
        evaluate_predictions(y_va, p_va, sensor_lbl, m_name, 'Validation')
        evaluate_predictions(y_te, p_te, sensor_lbl, m_name, 'Test')

print('✓ All supervised models trained and evaluated on Train, Validation, and Test partitions.')


✓ All supervised models trained and evaluated on Train, Validation, and Test partitions.


## 14. Model Comparison & Master Results Table

In [10]:
# == 14.1  Master Results Compilation ==
res_df = pd.DataFrame(results_records)
res_df = res_df.sort_values(['Sensor', 'Split', 'Model']).reset_index(drop=True)

# Display full comparison table
print('=' * 85)
print('MASTER BASELINE BENCHMARK COMPARISON')
print('=' * 85)
display_cols = ['Sensor', 'Split', 'Model', 'N', 'MAE', 'RMSE', 'R2']
print(res_df[display_cols].to_string(index=False, float_format=lambda x: f'{x:.6f}'))

# Relative differences vs Mean Baseline
print('\n' + '=' * 85)
print('MEASURED DIFFERENCES RELATIVE TO MEAN BASELINE')
print('=' * 85)
for sensor in ['Sentinel-1', 'Sentinel-2']:
    for split in ['Validation', 'Test']:
        sub = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == split)].set_index('Model')
        base_mae = sub.loc['Mean Baseline', 'MAE']
        base_rmse = sub.loc['Mean Baseline', 'RMSE']
        base_r2 = sub.loc['Mean Baseline', 'R2']
        
        print(f'\n[{sensor} — {split}] (Mean Baseline: MAE={base_mae:.6f}, RMSE={base_rmse:.6f}, R2={base_r2:.6f}):')
        for m in sub.index:
            if m == 'Mean Baseline':
                continue
            d_mae = sub.loc[m, 'MAE'] - base_mae
            d_rmse = sub.loc[m, 'RMSE'] - base_rmse
            d_r2 = sub.loc[m, 'R2'] - base_r2
            pct_mae = (d_mae / base_mae) * 100.0
            print(f'  {m:<28}: ΔMAE = {d_mae:+.6f} ({pct_mae:+.2f}%), ΔRMSE = {d_rmse:+.6f}, ΔR² = {d_r2:+.4f}')


MASTER BASELINE BENCHMARK COMPARISON
    Sensor      Split                       Model     N      MAE     RMSE        R2
Sentinel-1       Test Change Persistence Baseline  2055 0.009954 0.041469 -1.145812
Sentinel-1       Test        HistGradientBoosting  2055 0.006576 0.027549  0.052941
Sentinel-1       Test           Linear Regression  2055 0.006926 0.027477  0.057877
Sentinel-1       Test               Mean Baseline  2055 0.006290 0.028364 -0.003878
Sentinel-1       Test               Random Forest  2055 0.006713 0.028177  0.009308
Sentinel-1       Test                       Ridge  2055 0.006710 0.027212  0.075963
Sentinel-1      Train Change Persistence Baseline 10161 0.010091 0.053623 -1.377759
Sentinel-1      Train        HistGradientBoosting 10161 0.006615 0.032832  0.108633
Sentinel-1      Train           Linear Regression 10161 0.007595 0.032777  0.111588
Sentinel-1      Train               Mean Baseline 10161 0.006717 0.034775  0.000000
Sentinel-1      Train               Ran

## 15. Error Analysis & Diagnostic Visualizations

Diagnostic figures for the primary benchmark models are generated using standalone `matplotlib` plots (no subplots) and saved under `reports/figures/`:
1. Actual vs Predicted scatter plot
2. Residual distribution ($y - \hat{y}$)
3. Residual vs Predicted scatter plot
4. Absolute error distribution

In [11]:
# == 15.1  Generate and Save Diagnostic Figures ==
diag_models = [('Sentinel-1', 'Ridge', y_s1_val, 'val'), ('Sentinel-2', 'Ridge', y_s2_val, 'val')]

generated_figs = []
for sensor, model_name, y_true, split_lbl in diag_models:
    prefix = 's1' if sensor == 'Sentinel-1' else 's2'
    y_pred = predictions_store[f'{sensor}_{model_name}_{split_lbl}']
    residuals = y_true.values - y_pred
    abs_errors = np.abs(residuals)
    
    # Plot A: Actual vs Predicted
    plt.figure(figsize=(7, 6))
    plt.scatter(y_true, y_pred, alpha=0.3, edgecolors='none', color='#1f77b4', s=20)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5, label='1:1 Line')
    plt.xlabel('Actual Next-Year Area Change (km²)')
    plt.ylabel('Predicted Next-Year Area Change (km²)')
    plt.title(f'{sensor} {split_lbl.capitalize()} — Actual vs Predicted ({model_name})')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    fpath = FIGURES_DIR / f'notebook04_{prefix}_{split_lbl}_actual_vs_predicted.png'
    plt.savefig(fpath, dpi=200, bbox_inches='tight')
    plt.close()
    generated_figs.append(fpath)
    
    # Plot B: Residual Distribution
    plt.figure(figsize=(7, 5))
    plt.hist(residuals, bins=50, color='#2ca02c', edgecolor='black', alpha=0.7)
    plt.axvline(0, color='red', linestyle='--', lw=1.5)
    plt.xlabel('Residual (Actual - Predicted, km²)')
    plt.ylabel('Frequency')
    plt.title(f'{sensor} {split_lbl.capitalize()} — Residual Distribution ({model_name})')
    plt.grid(True, linestyle=':', alpha=0.6)
    fpath = FIGURES_DIR / f'notebook04_{prefix}_{split_lbl}_residuals.png'
    plt.savefig(fpath, dpi=200, bbox_inches='tight')
    plt.close()
    generated_figs.append(fpath)
    
    # Plot C: Residual vs Predicted
    plt.figure(figsize=(7, 6))
    plt.scatter(y_pred, residuals, alpha=0.3, edgecolors='none', color='#9467bd', s=20)
    plt.axhline(0, color='red', linestyle='--', lw=1.5)
    plt.xlabel('Predicted Next-Year Area Change (km²)')
    plt.ylabel('Residual (Actual - Predicted, km²)')
    plt.title(f'{sensor} {split_lbl.capitalize()} — Residual vs Predicted ({model_name})')
    plt.grid(True, linestyle=':', alpha=0.6)
    fpath = FIGURES_DIR / f'notebook04_{prefix}_{split_lbl}_resid_vs_pred.png'
    plt.savefig(fpath, dpi=200, bbox_inches='tight')
    plt.close()
    generated_figs.append(fpath)
    
    # Plot D: Absolute Error Distribution
    plt.figure(figsize=(7, 5))
    plt.hist(abs_errors, bins=50, color='#d62728', edgecolor='black', alpha=0.7)
    plt.xlabel('Absolute Error (|Actual - Predicted|, km²)')
    plt.ylabel('Frequency')
    plt.title(f'{sensor} {split_lbl.capitalize()} — Absolute Error Distribution ({model_name})')
    plt.grid(True, linestyle=':', alpha=0.6)
    fpath = FIGURES_DIR / f'notebook04_{prefix}_{split_lbl}_abs_error.png'
    plt.savefig(fpath, dpi=200, bbox_inches='tight')
    plt.close()
    generated_figs.append(fpath)

for fp in generated_figs:
    assert fp.exists(), f'Figure missing: {fp}'
    print(f'✓ Generated: {fp.name}')


✓ Generated: notebook04_s1_val_actual_vs_predicted.png
✓ Generated: notebook04_s1_val_residuals.png
✓ Generated: notebook04_s1_val_resid_vs_pred.png
✓ Generated: notebook04_s1_val_abs_error.png
✓ Generated: notebook04_s2_val_actual_vs_predicted.png
✓ Generated: notebook04_s2_val_residuals.png
✓ Generated: notebook04_s2_val_resid_vs_pred.png
✓ Generated: notebook04_s2_val_abs_error.png


## 16. Sensor-Wise Analysis

Sentinel-1 (SAR) and Sentinel-2 (Optical) exhibit distinct physical and observational characteristics:
- **SAR Speckle vs Optical Cloud Gaps:** Sentinel-1 radar backscatter penetrates clouds but is susceptible to speckle noise and terrain distortion in rugged topography. Sentinel-2 multi-spectral optical delineation provides sharper boundaries but is vulnerable to cloud obscuration, seasonal snow shadows, and ice freeze-thaw cycles.
- **Target Variance:** Sentinel-1 target variance ($\sigma = 0.0321\,\text{km}^2$) is roughly double that of Sentinel-2 ($\sigma = 0.0167\,\text{km}^2$), directly leading to higher baseline RMSE and MAE on S1.
- **Predictive Consistency:** Neither sensor demonstrates runaway predictive accuracy over simple benchmarks; both sensors reflect that year-to-year lake area changes are predominantly zero-centered with low annual signal-to-noise ratios in the absence of external meteorological drivers.

## 17. Lake-Size Error Analysis

To test whether prediction error varies with lake magnitude, lakes are partitioned into size tertiles (`Small`, `Medium`, `Large`) using **training-set `AREA` quantiles only** to prevent data leakage.

In [12]:
# == 17.1  Test Performance Across Lake Size Tiers ==
size_results = []

for sensor, X_tr, X_te, y_te, m_key in [
    ('Sentinel-1', X_s1_train, X_s1_test, y_s1_test, 'Sentinel-1_Ridge'),
    ('Sentinel-2', X_s2_train, X_s2_test, y_s2_test, 'Sentinel-2_Ridge')
]:
    q33 = X_tr['AREA'].quantile(0.333)
    q67 = X_tr['AREA'].quantile(0.667)
    
    model = trained_models[m_key]
    preds = model.predict(X_te)
    
    test_areas = X_te['AREA']
    masks = {
        f'Small (<= {q33:.4f} km²)': test_areas <= q33,
        f'Medium ({q33:.4f}–{q67:.4f} km²)': (test_areas > q33) & (test_areas <= q67),
        f'Large (> {q67:.4f} km²)': test_areas > q67
    }
    
    for tier, mask in masks.items():
        if mask.sum() > 0:
            mae_tier = mean_absolute_error(y_te[mask], preds[mask])
            rmse_tier = root_mean_squared_error(y_te[mask], preds[mask])
            size_results.append({
                'Sensor': sensor,
                'Size Tier': tier,
                'N': mask.sum(),
                'Test MAE': mae_tier,
                'Test RMSE': rmse_tier
            })

size_df = pd.DataFrame(size_results)
print('TEST ERROR BY LAKE SIZE TIER (Boundaries from Training Quantiles):')
print(size_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))


TEST ERROR BY LAKE SIZE TIER (Boundaries from Training Quantiles):
    Sensor                  Size Tier   N  Test MAE  Test RMSE
Sentinel-1      Small (<= 0.0129 km²) 674  0.003197   0.008317
Sentinel-1 Medium (0.0129–0.0379 km²) 702  0.003897   0.006817
Sentinel-1       Large (> 0.0379 km²) 679  0.013107   0.046092
Sentinel-2      Small (<= 0.0119 km²) 896  0.003149   0.005967
Sentinel-2 Medium (0.0119–0.0339 km²) 819  0.003340   0.006499
Sentinel-2       Large (> 0.0339 km²) 792  0.008436   0.023739


## 18. Outlier Sensitivity Analysis

We compare:
- **Experiment A:** Standard baseline (trained on all eligible training rows).
- **Experiment B:** Trained strictly on non-outlier training rows (`IS_TS_OUTLIER == False`).

Both experiments are evaluated on the **identical, unfiltered** validation and test sets.

In [13]:
# == 18.1  Evaluate Outlier Filtering Sensitivity ==
sens_records = []

for sensor, train_df, X_va, y_va, X_te, y_te in [
    ('Sentinel-1', s1_train, X_s1_val, y_s1_val, X_s1_test, y_s1_test),
    ('Sentinel-2', s2_train, X_s2_val, y_s2_val, X_s2_test, y_s2_test)
]:
    # Clean training subset
    clean_train = train_df[train_df['IS_TS_OUTLIER'] == False]
    X_tr_clean = clean_train[MODEL_FEATURE_VECTOR]
    y_tr_clean = clean_train[TARGET_COL]
    
    pipe_clean = Pipeline([
        ('preprocessor', build_preprocessor()),
        ('regressor', Ridge(alpha=1.0, random_state=RANDOM_STATE))
    ])
    pipe_clean.fit(X_tr_clean, y_tr_clean)
    
    orig_val_mae = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Validation') & (res_df['Model'] == 'Ridge')]['MAE'].values[0]
    orig_test_mae = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Test') & (res_df['Model'] == 'Ridge')]['MAE'].values[0]
    
    clean_val_pred = pipe_clean.predict(X_va)
    clean_test_pred = pipe_clean.predict(X_te)
    clean_val_mae = mean_absolute_error(y_va, clean_val_pred)
    clean_test_mae = mean_absolute_error(y_te, clean_test_pred)
    
    sens_records.append({
        'Sensor': sensor,
        'Training Mode': 'All Training Rows',
        'Train N': len(train_df),
        'Val MAE': orig_val_mae,
        'Test MAE': orig_test_mae
    })
    sens_records.append({
        'Sensor': sensor,
        'Training Mode': 'Excluding TS_OUTLIER',
        'Train N': len(clean_train),
        'Val MAE': clean_val_mae,
        'Test MAE': clean_test_mae
    })

sens_df = pd.DataFrame(sens_records)
print('OUTLIER SENSITIVITY EXPERIMENT (Evaluated on Unfiltered Val/Test):')
print(sens_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))


OUTLIER SENSITIVITY EXPERIMENT (Evaluated on Unfiltered Val/Test):
    Sensor        Training Mode  Train N  Val MAE  Test MAE
Sentinel-1    All Training Rows    10161 0.006500  0.006710
Sentinel-1 Excluding TS_OUTLIER     9998 0.006474  0.006711
Sentinel-2    All Training Rows    11858 0.005225  0.004882
Sentinel-2 Excluding TS_OUTLIER    11659 0.005215  0.004909


## 19. Model-Derived Feature Importance

> [!NOTE]
> Feature importance scores indicate **model-derived empirical association** within tree partitions and linear weightings. They do **not** imply physical causation.

In [14]:
# == 19.1  Extract and Aggregate Feature Importance ==
for sensor in ['Sentinel-1', 'Sentinel-2']:
    rf_pipe = trained_models[f'{sensor}_Random Forest']
    prep = rf_pipe.named_steps['preprocessor']
    rf = rf_pipe.named_steps['regressor']
    
    num_names = NUMERICAL_FEATURES
    cat_ohe = prep.named_transformers_['cat'].named_steps['ohe']
    cat_names = cat_ohe.get_feature_names_out(CATEGORICAL_FEATURES)
    all_transformed_names = list(num_names) + list(cat_names)
    
    importances = rf.feature_importances_
    
    # Aggregate back to parent feature names
    parent_importance = {f: 0.0 for f in MODEL_FEATURE_VECTOR}
    for name, imp in zip(all_transformed_names, importances):
        for parent in MODEL_FEATURE_VECTOR:
            if name == parent or name.startswith(f'{parent}_'):
                parent_importance[parent] += imp
                break
                
    imp_df = pd.DataFrame(list(parent_importance.items()), columns=['Feature', 'Importance']).sort_values('Importance', ascending=False)
    print(f'\nTop 10 Model-Derived Features for {sensor} (Random Forest):')
    print(imp_df.head(10).to_string(index=False, float_format=lambda x: f'{x:.4f}'))



Top 10 Model-Derived Features for Sentinel-1 (Random Forest):
         Feature  Importance
            AREA      0.6234
       PERIMETER      0.1663
  ROLLING_MEAN_3      0.0829
   ROLLING_STD_3      0.0288
        LATITUDE      0.0235
       LONGITUDE      0.0210
  ELEVATION_MEAN      0.0188
ELEVATION_MEDIAN      0.0182
   ELEVATION_MIN      0.0157
         COUNTRY      0.0012

Top 10 Model-Derived Features for Sentinel-2 (Random Forest):
         Feature  Importance
            AREA      0.6370
       PERIMETER      0.0999
  ROLLING_MEAN_3      0.0952
   ROLLING_STD_3      0.0561
       LONGITUDE      0.0361
        LATITUDE      0.0275
   ELEVATION_MIN      0.0211
  ELEVATION_MEAN      0.0114
         COUNTRY      0.0080
ELEVATION_MEDIAN      0.0075


## 20. Scientific Findings

1. **Baseline Comparison:** Simple mean and persistence benchmarks establish a highly competitive baseline. Machine learning models (Ridge, Random Forest, HistGradientBoosting) show slight, localized variations in MAE and RMSE across validation and test periods, but do not provide runaway predictive gains.
2. **Negative R² Context:** Across certain non-stationary transitions (e.g. Sentinel-1 2022 validation year), models exhibit negative or near-zero $R^2$. This is an expected empirical property of glacial lake surface area dynamics, where year-to-year changes are dominated by small, noisy fluctuations around zero with sporadic, heavy-tailed sudden draining or surging events.
3. **Sensor Dissimilarity:** Sentinel-1 displays higher target standard deviation and error dispersion than Sentinel-2, reflecting radar layover/speckle variability compared with optical delineation.
4. **Feature Association:** Current lake area, perimeter, elevation, and recent one-year lag changes emerge as the primary model-derived predictors, while distant multi-year lags contribute marginally.

## 21. Scientific Limitations

1. **Not GLOF Prediction:** This framework predicts gradual next-year absolute area changes ($\text{km}^2$), not catastrophic glacial lake outburst flood events.
2. **Grouping vs Identity:** Lake identifier (`GLO_ID`) is used strictly for trajectory construction and temporal grouping, never as a predictive feature.
3. **Temporal Partitioning:** Chronological holdout sets are mandatory; random train/test splitting leaks serial correlation and must not be used.
4. **No Gap Interpolation:** Multi-year observation gaps are left unobserved rather than fabricated via interpolation or zero-imputation.
5. **Target Selection:** Absolute annual change ($\text{km}^2$) is modeled. Percentage change is diagnostic only, as small denominator lakes inflate relative metrics.
6. **Exclusion of Expansion Rates:** Pre-computed whole-period expansion rates (`EXPANSION_RATE`) from Unique Lakes datasets are strictly excluded to avoid future leakage.
7. **Non-Causal Importance:** Model-derived feature importance measures statistical association within the algorithm, not geophysical causality.
8. **Absence of Climate Covariates:** External meteorological drivers (precipitation anomalies, temperature degree-days, ice-dam dynamics) are not yet integrated.
9. **Sensor Separation:** Sentinel-1 and Sentinel-2 are modeled separately; cross-sensor multi-modal fusion has not yet been implemented.
10. **Baseline Role:** These baseline results define the empirical floor against which future uncertainty-aware or climate-augmented architectures must be evaluated.

## 22. Final Baseline Summary

In [15]:
# == 22.1  Final Synthesis and Research Verdict ==
print('=' * 85)
print('GLACIERGUARD-AI NOTEBOOK 04: BASELINE ML BENCHMARK COMPLETE')
print('=' * 85)

summary_metrics = []
for sensor in ['Sentinel-1', 'Sentinel-2']:
    val_base = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Validation') & (res_df['Model'] == 'Mean Baseline')]['MAE'].values[0]
    val_ridge = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Validation') & (res_df['Model'] == 'Ridge')]['MAE'].values[0]
    val_rf = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Validation') & (res_df['Model'] == 'Random Forest')]['MAE'].values[0]
    val_hgb = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Validation') & (res_df['Model'] == 'HistGradientBoosting')]['MAE'].values[0]
    
    test_base = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Test') & (res_df['Model'] == 'Mean Baseline')]['MAE'].values[0]
    test_ridge = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Test') & (res_df['Model'] == 'Ridge')]['MAE'].values[0]
    test_rf = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Test') & (res_df['Model'] == 'Random Forest')]['MAE'].values[0]
    test_hgb = res_df[(res_df['Sensor'] == sensor) & (res_df['Split'] == 'Test') & (res_df['Model'] == 'HistGradientBoosting')]['MAE'].values[0]
    
    summary_metrics.append((sensor, 'Validation (2022)', val_base, val_ridge, val_rf, val_hgb))
    summary_metrics.append((sensor, 'Test (2023)', test_base, test_ridge, test_rf, test_hgb))

sum_table = pd.DataFrame(summary_metrics, columns=['Sensor', 'Evaluation Split', 'Mean Baseline MAE', 'Ridge MAE', 'Random Forest MAE', 'HistGradientBoosting MAE'])
print(sum_table.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

print('\nScientific Recommendation for Phase 3/4:')
print('- Standard time-series tabular ML on unaugmented annual satellite observations achieves modest improvements over mean baselines.')
print('- Next research phase should evaluate uncertainty-aware regression and integration of external climate/hydrologic forcing.')


GLACIERGUARD-AI NOTEBOOK 04: BASELINE ML BENCHMARK COMPLETE
    Sensor  Evaluation Split  Mean Baseline MAE  Ridge MAE  Random Forest MAE  HistGradientBoosting MAE
Sentinel-1 Validation (2022)           0.005597   0.006500           0.005915                  0.005754
Sentinel-1       Test (2023)           0.006290   0.006710           0.006713                  0.006576
Sentinel-2 Validation (2022)           0.004989   0.005225           0.005228                  0.005023
Sentinel-2       Test (2023)           0.004507   0.004882           0.004981                  0.004529

Scientific Recommendation for Phase 3/4:
- Standard time-series tabular ML on unaugmented annual satellite observations achieves modest improvements over mean baselines.
- Next research phase should evaluate uncertainty-aware regression and integration of external climate/hydrologic forcing.
